# Lasso Regression (L1 Regularization) - California Housing Dataset

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

**Steps:**
1. **df.shape**: Check the number of rows (samples) and columns (features)
2. **df.info()**: Get information about data types, non-null counts, and memory usage
3. **df.describe()**: View statistical summary (mean, std, min, max, quartiles) for each feature

This helps us understand:
- The size of our dataset
- Data types of each column
- Whether there are any missing values
- The range and distribution of each feature

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

Missing values can significantly impact model performance. We need to identify and handle them appropriately.

**Steps:**
- Use `df.isnull().sum()` to count missing values in each column
- If missing values exist, we can either:
  - Drop rows with missing values
  - Impute missing values (mean, median, mode, or using ML algorithms)

In this dataset, we can see there are no missing values, so no action is needed.

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

Duplicate rows can bias our model by giving more weight to certain observations. We need to identify and remove them.

**Steps:**
- Use `df.duplicated().sum()` to count duplicate rows
- If duplicates exist, use `df.drop_duplicates()` to remove them

In this dataset, there are no duplicate rows, so no action is needed.

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

We need to separate our data into:
- **X (Features)**: All columns except the target variable (Price)
- **y (Target)**: The variable we want to predict (Price)

**Steps:**
- Use `df.drop("Price", axis=1)` to get features (X)
- Use `df["Price"]` to get target (y)

This separation is essential for training our machine learning model.

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

Visualizing the distribution of each feature helps us understand:
- The shape of the data (normal, skewed, etc.)
- Potential outliers
- Whether features need transformation

**Steps:**
- Use `df.hist()` to create histograms for all numerical features
- Set appropriate figure size and number of bins for better visualization

This helps identify which features might need scaling or transformation.

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

Understanding the relationships between features and the target variable helps in feature selection and understanding the data.

**Steps:**
1. Calculate correlation matrix using `df.corr()`
2. Sort correlations with the target variable (Price) to identify most important features
3. Create a heatmap visualization to easily see correlations

**Key Insights:**
- High positive correlation with Price means the feature increases house price
- High negative correlation means the feature decreases house price
- Features with low correlation might be less important for prediction

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

Boxplots help identify outliers in each feature. Outliers can significantly affect model performance.

**Steps:**
- Create boxplots for each feature using `plt.boxplot()`
- Examine each plot for points outside the whiskers (potential outliers)

**Interpretation:**
- Points outside the whiskers are considered outliers
- We may need to handle outliers by:
  - Removing them (if they're errors)
  - Capping them (winsorization)
  - Using robust models that handle outliers well

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Standardization)

Feature scaling is crucial for algorithms that use distance-based calculations or gradient descent. It ensures all features contribute equally to the model.

**Why Scale Features?**
- Features have different units and scales (e.g., MedInc in thousands, Latitude in degrees)
- Without scaling, features with larger values can dominate the model
- Helps convergence in gradient-based algorithms

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

Where:
- **z** = standardized value
- **x** = original value
- **μ** = mean of the feature
- **σ** = standard deviation of the feature

**Steps:**
1. Initialize StandardScaler
2. Fit and transform the features using `fit_transform()`
3. Convert back to DataFrame with original column names

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

Splitting the data into training and testing sets is essential to evaluate model performance on unseen data.

**Why Split?**
- Training set: Used to train the model (learn patterns)
- Testing set: Used to evaluate how well the model generalizes to new data
- Prevents overfitting and provides unbiased performance metrics

**Steps:**
1. Use `train_test_split()` to split the data
2. Set `test_size=0.2` (20% for testing, 80% for training)
3. Set `random_state=42` for reproducibility
4. Verify the shapes of train and test sets

**Common Splits:**
- 80-20 (default): Good balance for most datasets
- 70-30: When you have more data and want more for testing
- 90-10: When you have limited data and need more for training

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## Lasso Regression (L1 Regularization) Formula

**The Lasso Regression Equation:**

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n + \epsilon$$

**The Lasso Loss Function (with L1 Regularization):**

$$L(\beta) = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p} |\beta_j|$$

Where:
- **y** = dependent variable (target/price)
- **x₁, x₂, ..., xₙ** = independent variables (features)
- **β₀, β₁, ..., βₙ** = regression coefficients
- **λ (lambda/alpha)** = regularization parameter (controls penalty strength)
- **ε** = error term

**Key Concepts:**
- Lasso regression adds an L1 penalty term (λ∑|βⱼ|) to the ordinary least squares loss
- This penalty can shrink coefficients to exactly zero, performing feature selection
- Helps prevent overfitting by reducing model complexity
- The regularization parameter (α/λ) controls the strength of the penalty:
  - α = 0: Same as ordinary linear regression (no regularization)
  - α → ∞: All coefficients approach zero (strong regularization)
- Lasso is particularly useful when you have many features and want automatic feature selection
- Features with zero coefficient are effectively removed from the model

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
lasso = Lasso(alpha=1.0)

lasso.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = lasso.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R²:",r2)

## Coefficient Interpretation

In Lasso regression, coefficients can be shrunk to exactly zero, which is the key feature selection property of L1 regularization.

**Key Points:**
- Coefficients that are exactly zero indicate that the corresponding features are not important
- This automatic feature selection helps identify the most relevant features
- The degree of shrinkage depends on the alpha parameter

In [ ]:
coef_df = pd.DataFrame({

    "Feature":X_train.columns,

    "Coefficient":lasso.coef_

})

coef_df.sort_values(
    by="Coefficient",
    ascending=False
)
print(
    "Intercept:",
    lasso.intercept_
)

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()]
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title(
    "Lasso Regression"
)

plt.show()

## Residual Plot

In [ ]:
residuals = y_test-y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(y=0)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Residuals"
)

plt.title(
    "Residual Plot"
)

plt.show()

## Alpha Parameter Tuning

The regularization parameter (α) controls the trade-off between:
- **Bias**: Higher α increases bias (coefficients shrink more)
- **Variance**: Higher α reduces variance (less overfitting)

We test different α values to find the optimal balance:
- α = 0.01: Very weak regularization (similar to OLS)
- α = 0.1: Weak regularization
- α = 1.0: Moderate regularization (default)
- α = 10: Strong regularization
- α = 100: Very strong regularization

The best α is chosen based on cross-validation performance metrics.

In [ ]:
alphas=[0.01,0.1,1,10,100]

for a in alphas:

    model=Lasso(alpha=a)

    model.fit(
        X_train,
        y_train
    )

    pred=model.predict(
        X_test
    )

    r2=r2_score(
        y_test,
        pred
    )

    print(
        f"Alpha {a}: R² = {r2:.4f}, Non-zero coefficients: {np.sum(model.coef_ != 0)}"
    )

## Summary

Lasso regression with L1 regularization provides:
- Automatic feature selection through coefficient shrinkage to zero
- Prevention of overfitting through regularization
- Interpretability by identifying the most important features
- Better performance when dealing with high-dimensional data with many irrelevant features